In [ ]:
import pymongo
import pandas as pd
import json
import db.knowhere_db as kdb

In [ ]:
reader = kdb.Reader(db_name='knowhere')
data = reader.get_dataframe_pivoted('iphone_test3', username='emil', 
                                    min_date = '2017-03-19', max_date = '2017-03-20')
data.head()

### Plotting Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn
from datetime import datetime

In [ ]:
def get_daily_data(day, user_id = kdb.ObjectId('58c5b91d71a9e82f33ad1669'), sensor = "GPS", collection = "iphone_test"):
    to_plot = reader.get_dataframe_pivoted(collection = collection, user_id = user_id, sensor = sensor)
    to_plot = to_plot.ix[day]
    to_plot = to_plot.astype(float)
    #plt.plot(to_plot['GPS (Latitude)'], to_plot['GPS (Longitude)'], zorder = 2)
    plt.scatter(to_plot['GPS (Latitude)'], to_plot['GPS (Longitude)'], zorder = 1, c = to_plot.index)
    plt.xlabel("Latitude")
    plt.ylabel("Longitude")
    plt.title("Location Data for %s" % (day))
    

### Activity Measurement

In [ ]:
t1 = reader.get_dataframe_pivoted(collection = 'iphone_test', sensor = "Acceleration")

In [ ]:
t1 = t1.astype(float)

In [ ]:
t1.plot()

In [ ]:
from math import sqrt

accel_mag = []
for row in range(len(t1)):
    accel_mag.append(sqrt(t1.iloc[row, 0]**2 + t1.iloc[row, 1]**2 + t1.iloc[row, 2]**2))
    
t1['accel_mag'] = accel_mag

In [ ]:
#check acceleration levels are various percentiles, just to see what's there
accel_thresholds = []

for quan in np.linspace(0, 1.0, 9):
    accel_thresholds.append((quan, t1['accel_mag'].quantile(q=quan)))
    
accel_thresholds

In [ ]:
#set a threshold based on 1 day of Emil's data, ideally we'll set this based off of all user data
activity_threshold = 0.051845671879112265

In [ ]:
def check_single_active(accel_mag, activity_threshold = 0.051845671879112265):
    if accel_mag >= activity_threshold:
        return 1
    else:
        return 0
    return 0


In [ ]:
def check_active(mag_list):
    mag_list = list(mag_list)
    activity_raw = 0
    for active in mag_list:
        activity_raw += check_single_active(active)
    return float(activity_raw) / len(t1['accel_mag'])

In [ ]:
check_active(t1['accel_mag'])

# Emil Look Here!

### Distance Traveled

In [ ]:
#Haversine Function, pulled from StackOverflow

def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees, AKA Latitude / Longitude)

    All args must be of equal length.    

    """
    from math import radians, cos, sin, asin, sqrt

    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    #haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2

    c = 2 * np.arcsin(np.sqrt(a))
    miles = 3956 * c # Radius of earth in miles. Use 6370 for kilometers
    return miles

In [ ]:
def distance_traveled(min_date, max_date, username):
    """
    Calculate the distance a user has traveled over a given date range. Involves all modes of transportation, 
    and is calculated using Great-circle distance.
    """
    
    #make DB query to get GPS data for specified day 
    day_dist = reader.get_dataframe_pivoted(collection = 'iphone_test3', sensor = "GPS", username = username,
                                min_date = min_date, max_date = max_date)
    day_dist = day_dist.astype(float)
    
    #Use haversine function to get distances between all GPS points
    dist_traveled = haversine(day_dist['GPS Longitude'].shift(), day_dist['GPS Latitude'].shift(), 
          day_dist.ix[1:, 'GPS Longitude'], day_dist.ix[1:, 'GPS Latitude'])
    
    return dist_traveled.sum()

In [ ]:
distance_traveled("03-01-2017", "03-20-2017", "glen")

In [ ]:
def daily_distance_traveled(day, username):
    #Get day after day passed into function so that DB call works
    import pandas.tseries.offsets as pdo
    next_day = pd.to_datetime(day) + pdo.Day()
    
    #make DB query to get GPS data for specified day 
    day_dist = reader.get_dataframe_pivoted(collection = 'iphone_test3', sensor = "GPS", username = username,
                                min_date = str(day), max_date = str(next_day))
    day_dist = day_dist.astype(float)
    
    dist_traveled = haversine(day_dist['GPS Longitude'].shift(), day_dist['GPS Latitude'].shift(), 
          day_dist.ix[1:, 'GPS Longitude'], day_dist.ix[1:, 'GPS Latitude'])
    dt = dist_traveled.sum()
    return dt

In [ ]:
daily_distance_traveled('03/19/2017', 'glen')

In [ ]:
def daily_distance_traveled_with_time(start_day, end_day, username):
    #make DB query to get GPS data for specified day 
    day_dist = reader.get_dataframe_pivoted(collection = 'iphone_test3', sensor = "GPS", username = username,
                                min_date = start_day, max_date = end_day)

    day_dist = day_dist.astype(float)
    
    hourly = day_dist.groupby(pd.Grouper(freq='1H'))
    hourly_distances = []

    for hour in hourly:
        h = hour[0]
        print h
        distance = haversine(hour[1]['GPS Longitude'].shift(), hour[1]['GPS Latitude'].shift(), 
                   hour[1].ix[1:, 'GPS Longitude'], hour[1].ix[1:, 'GPS Latitude']).sum()
        print distance
        hourly_distances.append((h, distance))

    return hourly_distances

In [ ]:
distance_traveled("03-09-2017", "03-13-2017", "emil")

In [ ]:
dist = 0
for x in daily_distance_traveled_with_time('03/09/2017','03/13/2017', 'emil'):
    dist += x[1]
    
dist

# Emil, stop looking!

### Orientation Calculator


In [ ]:
t2 = reader.get_dataframe_pivoted(collection = 'iphone_test', sensor = 'Compass', username = 'emil')

In [ ]:
def daily_orientation(day):
    daily_orientation = {"N": 0, "E": 0, "S": 0, "W": 0}
    for index in t2.ix[day].itertuples():
        orientation = float(index[2])
        if (orientation > 305) or (orientation < 45):
            d_o_add("N", daily_orientation)
        elif (orientation >= 45) and (orientation < 135):
            d_o_add("E", daily_orientation)
        elif (orientation >= 135) and (orientation < 225):
            d_o_add("S", daily_orientation)
        else:
            d_o_add("W", daily_orientation)
    return daily_orientation

In [ ]:
def d_o_add(direction, daily_orientation):
    old_val = daily_orientation[direction]
    daily_orientation[direction] = (old_val + 1)
    return daily_orientation

In [ ]:
daily_orientation("03-10-2017")

### Subway Identifier

In [ ]:
full_emil = reader.get_dataframe_pivoted(collection = 'iphone_test', username = 'emil')

In [ ]:
sub = full_emil.iloc[:, 0:5]
sub = sub.astype(float)

In [ ]:
sub.iloc[:, 3:4].plot()

In [ ]:
def add_acceleration_magnitude(df):
    "Adds Acceleration magnitude to a dataframe. First 3 columns must be acceleration vectors. Must be 3D."
    from math import sqrt

    accel_mag = []
    
    for row in range(len(df)):
        accel_mag.append(sqrt(df.iloc[row, 0]**2 + df.iloc[row, 1]**2 + df.iloc[row, 2]**2))

    df['accel_mag'] = accel_mag
    return 'finished!'

In [ ]:
add_acceleration_magnitude(sub)

In [ ]:
sub['accel_mag'].plot()